In [2]:
import ee 
import geemap
ee.Authenticate()
ee.Initialize()


# Create map
Map = geemap.Map(center=[8.5, -80], zoom=7)

# Panama boundary
countries = ee.FeatureCollection("FAO/GAUL/2015/level0")

panama = countries.filter(
    ee.Filter.eq("ADM0_NAME", "Panama")
)

# Protected areas
pa = ee.FeatureCollection("WCMC/WDPA/current/polygons")

# Clip protected areas to Panama
pa_panama = pa.filterBounds(panama.geometry())

# Rasterize protected areas
pa_raster = ee.Image().byte().paint(
    featureCollection=pa_panama,
    color=1
).clip(panama)

# Compute distance to nearest protected area (meters)
distance_to_pa = (
    pa_raster
    .distance(ee.Kernel.euclidean(50000, 'meters'))
    .rename('distance_to_pa')
)

# Visualization parameters
pa_vis = {
    'palette': ['006400']
}

dist_vis = {
    'min': 0,
    'max': 50000,
    'palette': ['white', 'yellow', 'orange', 'red']
}

# Add layers
Map.addLayer(panama, {}, 'Panama Boundary')
Map.addLayer(pa_panama, {'color': 'green'}, 'Protected Areas')
Map.addLayer(pa_raster, pa_vis, 'PA Raster')
Map.addLayer(distance_to_pa, dist_vis, 'Distance to Protected Areas')

# Center map
Map.centerObject(panama, 7)

# Display map
Map

Map(center=[8.5158389458998, -80.10966640141521], controls=(WidgetControl(options=['position', 'transparent_bg…